# PDF Compression for Claude Project Ingestion

This notebook compresses large PDF slide decks so they meet Claude's project file size limit (~10 MB per file).

**Workflow:**
1. Read all PDFs from the notebook directory and report their sizes
2. Flag files that exceed the Claude ingestion limit
3. Compress each file using the best available method (Ghostscript → pikepdf → PyPDF2)
4. Report before/after sizes and compression ratios

In [1]:
# ── Pre-flight: install / verify all required libraries ──────────────────────
import subprocess, sys, shutil, importlib, importlib.util, importlib.metadata

def _pkg_version(pkg_import_name: str) -> str | None:
    """Return installed version string, or None if not importable."""
    if importlib.util.find_spec(pkg_import_name) is None:
        return None
    dist_map = {"pikepdf": "pikepdf", "PyPDF2": "PyPDF2", "fitz": "pymupdf"}
    try:
        return importlib.metadata.version(dist_map.get(pkg_import_name, pkg_import_name))
    except Exception:
        return "installed (version unknown)"

def _pip_install(package: str) -> tuple[bool, str]:
    """Install via pip; invalidate import caches so the package is importable immediately."""
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", package, "--quiet", "--upgrade"],
        capture_output=True, text=True,
    )
    importlib.invalidate_caches()
    if result.returncode != 0:
        return False, result.stderr.strip()[:200]
    return True, ""

REQUIRED = [
    ("fitz",   "pymupdf", "Page re-rendering at target DPI — best pure-Python compression for image-heavy slides"),
    ("pikepdf", "pikepdf", "Advanced PDF compression — stream recompression + deduplication"),
    ("PyPDF2",  "PyPDF2",  "Page-level content stream compression (fallback)"),
]

print(f"{'Package':<12} {'Status':<14} {'Version':<22} Notes")
print("-" * 90)

for import_name, pip_name, notes in REQUIRED:
    ver = _pkg_version(import_name)
    if ver:
        print(f"{pip_name:<12} {'OK':<14} {ver:<22} {notes}")
    else:
        print(f"{pip_name:<12} {'Installing...':<14} {'':22} {notes}", end="", flush=True)
        ok, err = _pip_install(pip_name)
        ver = _pkg_version(import_name) or "?"
        if ok:
            print(f"\r{pip_name:<12} {'Installed':<14} {ver:<22} {notes}")
        else:
            print(f"\r{pip_name:<12} {'FAILED':<14} {'':22} ERROR: {err}")

# ── Ghostscript (OS-level tool — not pip-installable) ─────────────────────────
import glob as _glob

def _find_ghostscript() -> str | None:
    """Return the GS executable path if Ghostscript is available, else None."""
    for exe in ("gswin64c", "gswin32c", "gs"):
        found = shutil.which(exe)
        if found:
            return found
    for pattern in [
        r"C:\Program Files\gs\gs*\bin\gswin64c.exe",
        r"C:\Program Files (x86)\gs\gs*\bin\gswin32c.exe",
    ]:
        matches = _glob.glob(pattern)
        if matches:
            return sorted(matches)[-1]
    return None

gs = _find_ghostscript()
print(f"\n{'Ghostscript':<12} {'OK (' + gs + ')' if gs else 'NOT FOUND':<45} Best compressor for image-heavy slides")
if not gs:
    print("""
  Ghostscript not found — notebook will use pymupdf (good fallback).
  For best results install it:
    Windows : https://www.ghostscript.com/releases/gsdnld.html
    macOS   : brew install ghostscript
    Linux   : sudo apt install ghostscript
""")


Package      Status         Version                Notes
------------------------------------------------------------------------------------------
pymupdf      OK             1.27.2.3               Page re-rendering at target DPI — best pure-Python compression for image-heavy slides
pikepdf      OK             10.7.1                 Advanced PDF compression — stream recompression + deduplication
PyPDF2       OK             3.0.1                  Page-level content stream compression (fallback)

Ghostscript  NOT FOUND                                     Best compressor for image-heavy slides

  Ghostscript not found — notebook will use pymupdf (good fallback).
  For best results install it:
    Windows : https://www.ghostscript.com/releases/gsdnld.html
    macOS   : brew install ghostscript
    Linux   : sudo apt install ghostscript



In [2]:
import os
import pathlib
import fitz       # pymupdf
import pikepdf
from PyPDF2 import PdfReader, PdfWriter

print("All libraries imported successfully.")
print(f"  pymupdf : {fitz.__version__}")
print(f"  pikepdf : {pikepdf.__version__}")
print(f"  PyPDF2  : {__import__('PyPDF2').__version__}")


All libraries imported successfully.
  pymupdf : 1.27.2.3
  pikepdf : 10.7.1
  PyPDF2  : 3.0.1


In [3]:
# ── Configuration ────────────────────────────────────────────────────────────

# Directory containing the PDFs (same folder as this notebook)
PDF_DIR = pathlib.Path(r"C:\Users\User\OneDrive\Desktop\MBA Data and AI\Year 2\Deep Learning")

# Where compressed files will be saved
OUTPUT_DIR = PDF_DIR / "compressed"
OUTPUT_DIR.mkdir(exist_ok=True)

# Claude project knowledge-file limit (MB)
CLAUDE_LIMIT_MB = 10.0

# Only compress files larger than this threshold (skip already-small files)
COMPRESS_THRESHOLD_MB = 5.0

# pymupdf compression settings (used when Ghostscript is not available)
# Lower DPI or JPEG quality = smaller file, lower visual quality
PYMUPDF_DPI          = 120   # try 96 for more aggressive compression
PYMUPDF_JPEG_QUALITY = 75    # 0–100; try 60 for more aggressive compression

print(f"PDF source       : {PDF_DIR}")
print(f"Output dir       : {OUTPUT_DIR}")
print(f"Claude limit     : {CLAUDE_LIMIT_MB} MB per file")
print(f"pymupdf DPI      : {PYMUPDF_DPI}")
print(f"pymupdf JPEG Q   : {PYMUPDF_JPEG_QUALITY}")


PDF source       : C:\Users\User\OneDrive\Desktop\MBA Data and AI\Year 2\Deep Learning
Output dir       : C:\Users\User\OneDrive\Desktop\MBA Data and AI\Year 2\Deep Learning\compressed
Claude limit     : 10.0 MB per file
pymupdf DPI      : 120
pymupdf JPEG Q   : 75


## Section 1: Read PDFs & Check Sizes

Scan the source directory for all PDF files, report their sizes, and flag any that exceed Claude's ingestion limit.

In [4]:
def get_size_mb(path: pathlib.Path) -> float:
    return path.stat().st_size / (1024 * 1024)

def scan_pdfs(directory: pathlib.Path) -> list[dict]:
    """Return a list of dicts with file metadata for every PDF in directory."""
    pdfs = sorted(directory.glob("*.pdf"))
    records = []
    for p in pdfs:
        size_mb = get_size_mb(p)
        needs_compression = size_mb > COMPRESS_THRESHOLD_MB
        within_limit = size_mb <= CLAUDE_LIMIT_MB
        records.append({
            "file": p,
            "name": p.name,
            "size_mb": size_mb,
            "needs_compression": needs_compression,
            "within_limit": within_limit,
        })
    return records

pdf_records = scan_pdfs(PDF_DIR)

# ── Pretty-print the inventory ────────────────────────────────────────────────
COL = 60
print(f"{'File':<{COL}} {'Size (MB)':>10}  {'Claude OK?':>10}  {'Action':>12}")
print("-" * (COL + 38))
for r in pdf_records:
    status  = "YES" if r["within_limit"] else "NO  <-- OVER LIMIT"
    action  = "skip" if not r["needs_compression"] else "compress"
    print(f"{r['name']:<{COL}} {r['size_mb']:>10.2f}  {status:>10}  {action:>12}")

over_limit = [r for r in pdf_records if not r["within_limit"]]
print(f"\n{len(pdf_records)} PDF(s) found. {len(over_limit)} exceed the {CLAUDE_LIMIT_MB} MB Claude limit.")

File                                                          Size (MB)  Claude OK?        Action
--------------------------------------------------------------------------------------------------
20260330_l01_deep-learning.pdf                                    22.15  NO  <-- OVER LIMIT      compress
20260413_l02_deep-learning.pdf                                    33.67  NO  <-- OVER LIMIT      compress
20260504_l03_deep-learning.pdf                                    37.85  NO  <-- OVER LIMIT      compress
20260518_l04_deep-learning.pdf                                    66.91  NO  <-- OVER LIMIT      compress
Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques-to-Build-Intelligent-Systems-O’Reilly-Media-2019.pdf      31.52  NO  <-- OVER LIMIT      compress

5 PDF(s) found. 5 exceed the 10.0 MB Claude limit.


## Section 2: Compression Functions

Three compression methods are tried in order of effectiveness:

| Method | How it works | Best for |
|--------|-------------|----------|
| **Ghostscript** | Re-renders pages + resamples images | Image-heavy slide PDFs — greatest size reduction |
| **pikepdf** | Lossless stream compression + object deduplication | Text/vector PDFs |
| **PyPDF2** | `compress_content_streams()` on each page | Fallback; minimal gains on image PDFs |

The pipeline tries Ghostscript first; if it is not installed it falls back automatically.

In [5]:
import shutil

# ── Ghostscript helper ─────────────────────────────────────────────────────────

def _find_ghostscript() -> str | None:
    """Return the GS executable path if Ghostscript is available, else None."""
    for exe in ("gswin64c", "gswin32c", "gs"):
        found = shutil.which(exe)
        if found:
            return found
    for pattern in [
        r"C:\Program Files\gs\gs*\bin\gswin64c.exe",
        r"C:\Program Files (x86)\gs\gs*\bin\gswin32c.exe",
    ]:
        matches = _glob.glob(pattern)
        if matches:
            return sorted(matches)[-1]
    return None

def compress_ghostscript(input_path: pathlib.Path, output_path: pathlib.Path,
                          dpi: int = 150) -> tuple[bool, str]:
    gs = _find_ghostscript()
    if gs is None:
        return False, "Ghostscript not found"
    cmd = [
        gs, "-sDEVICE=pdfwrite", "-dCompatibilityLevel=1.5",
        "-dPDFSETTINGS=/ebook", "-dNOPAUSE", "-dQUIET", "-dBATCH",
        f"-dColorImageResolution={dpi}", f"-dGrayImageResolution={dpi}",
        f"-dMonoImageResolution={dpi}", f"-sOutputFile={output_path}", str(input_path),
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if result.returncode != 0:
            return False, f"Ghostscript error: {result.stderr.strip()[:200]}"
        return True, "Ghostscript"
    except subprocess.TimeoutExpired:
        return False, "Ghostscript timed out"
    except Exception as e:
        return False, f"Ghostscript exception: {e}"


# ── pymupdf helper (re-render each page as JPEG at target DPI) ─────────────────

def compress_pymupdf(input_path: pathlib.Path, output_path: pathlib.Path,
                     dpi: int = PYMUPDF_DPI,
                     jpeg_quality: int = PYMUPDF_JPEG_QUALITY) -> tuple[bool, str]:
    try:
        import fitz as _fitz
    except ImportError:
        return False, "pymupdf not installed — run: pip install pymupdf"
    try:
        src = _fitz.open(str(input_path))
        out = _fitz.open()
        zoom = dpi / 72.0
        mat = _fitz.Matrix(zoom, zoom)
        for page in src:
            pix = page.get_pixmap(matrix=mat, alpha=False)
            img_bytes = pix.tobytes(output="jpeg", jpg_quality=jpeg_quality)
            new_page = out.new_page(width=pix.width, height=pix.height)
            new_page.insert_image(new_page.rect, stream=img_bytes, keep_proportion=False)
        out.save(str(output_path), garbage=4, deflate=True, clean=True)
        src.close()
        out.close()
        return True, "pymupdf"
    except Exception as e:
        return False, f"pymupdf error: {e}"


# ── pikepdf helper ─────────────────────────────────────────────────────────────

def compress_pikepdf(input_path: pathlib.Path, output_path: pathlib.Path) -> tuple[bool, str]:
    try:
        with pikepdf.open(input_path) as pdf:
            pdf.save(output_path, compress_streams=True, recompress_flate=True,
                     object_stream_mode=pikepdf.ObjectStreamMode.generate)
        return True, "pikepdf"
    except Exception as e:
        return False, f"pikepdf error: {e}"


# ── PyPDF2 helper ──────────────────────────────────────────────────────────────

def compress_pypdf2(input_path: pathlib.Path, output_path: pathlib.Path) -> tuple[bool, str]:
    try:
        reader = PdfReader(str(input_path))
        writer = PdfWriter()
        for page in reader.pages:
            page.compress_content_streams()
            writer.add_page(page)
        with open(output_path, "wb") as f:
            writer.write(f)
        return True, "PyPDF2"
    except Exception as e:
        return False, f"PyPDF2 error: {e}"


# ── Unified compress pipeline ──────────────────────────────────────────────────

def compress_pdf(input_path: pathlib.Path, output_path: pathlib.Path,
                 verbose: bool = True) -> dict:
    size_before = get_size_mb(input_path)
    result = {
        "input": input_path.name,
        "output": output_path,
        "size_before_mb": size_before,
        "size_after_mb": None,
        "reduction_pct": None,
        "method": None,
        "success": False,
        "error": None,
    }

    attempts = [
        ("Ghostscript", compress_ghostscript),
        ("pymupdf",     compress_pymupdf),
        ("pikepdf",     compress_pikepdf),
        ("PyPDF2",      compress_pypdf2),
    ]

    for method_name, fn in attempts:
        tmp_out = output_path.with_suffix(".tmp.pdf")
        ok, msg = fn(input_path, tmp_out)

        if ok and tmp_out.exists():
            size_after = get_size_mb(tmp_out)
            if size_after < size_before:
                tmp_out.replace(output_path)
                result.update(success=True, method=msg, size_after_mb=size_after,
                               reduction_pct=round((1 - size_after / size_before) * 100, 1))
                if verbose:
                    print(f"\n    ✓ {method_name}: {size_before:.2f} → {size_after:.2f} MB ({result['reduction_pct']}% saved)")
                return result
            else:
                tmp_out.unlink(missing_ok=True)
                if verbose:
                    print(f"\n    ✗ {method_name}: not smaller ({size_after:.2f} MB >= {size_before:.2f} MB)")
        else:
            if tmp_out.exists():
                tmp_out.unlink(missing_ok=True)
            if verbose:
                print(f"\n    ✗ {method_name}: FAILED — {msg}")
            result["error"] = msg

    return result


gs_exe = _find_ghostscript()
print(f"Ghostscript : {'YES (' + gs_exe + ')' if gs_exe else 'NOT FOUND'}")
print(f"Pipeline    : Ghostscript → pymupdf ({PYMUPDF_DPI} DPI / Q{PYMUPDF_JPEG_QUALITY}) → pikepdf → PyPDF2")
print("Compression functions defined.")


Ghostscript : NOT FOUND
Pipeline    : Ghostscript → pymupdf (120 DPI / Q75) → pikepdf → PyPDF2
Compression functions defined.


## Section 3: Compress & Report Results

Run the compression pipeline on every PDF that needs it, then display a summary table showing:
- Original vs. compressed size
- Reduction percentage
- Whether the output is now within Claude's limit
- Any errors encountered

In [6]:
results = []

for rec in pdf_records:
    src = rec["file"]
    dst = OUTPUT_DIR / src.name

    if not rec["needs_compression"]:
        print(f"[SKIP]  {src.name}  ({rec['size_mb']:.2f} MB — already small enough)")
        results.append({**rec, "method": "skipped", "success": True,
                        "size_after_mb": rec["size_mb"], "reduction_pct": 0.0, "error": None})
        continue

    print(f"[COMPRESS]  {src.name}  ({rec['size_mb']:.2f} MB) ...", end=" ", flush=True)
    res = compress_pdf(src, dst)
    results.append(res)

    if res["success"]:
        claude_ok = res["size_after_mb"] <= CLAUDE_LIMIT_MB
        flag = "" if claude_ok else "  <-- still over limit"
        print(f"done  [{res['method']}]  {res['size_before_mb']:.2f} → {res['size_after_mb']:.2f} MB  (-{res['reduction_pct']}%){flag}")
    else:
        print(f"FAILED  {res['error']}")

# ── Summary table ──────────────────────────────────────────────────────────────
print("\n" + "=" * 95)
print(f"{'File':<55} {'Before':>8} {'After':>8} {'Saved%':>7} {'Method':<14} {'Claude OK?':>10}")
print("=" * 95)

for r in results:
    after  = f"{r['size_after_mb']:.2f}" if r["size_after_mb"] is not None else "N/A"
    saved  = f"-{r['reduction_pct']}%" if r.get("reduction_pct") else "—"
    method = r.get("method") or "—"
    if r.get("error") and not r.get("success"):
        claude_ok = "ERROR"
    elif r["size_after_mb"] is not None:
        claude_ok = "YES" if r["size_after_mb"] <= CLAUDE_LIMIT_MB else "NO"
    else:
        claude_ok = "—"

    name = r["input"] if "input" in r else r["name"]
    print(f"{name:<55} {r['size_before_mb']:>8.2f} {after:>8} {saved:>7} {method:<14} {claude_ok:>10}")

print("=" * 95)

# ── Final status ───────────────────────────────────────────────────────────────
failures  = [r for r in results if not r.get("success")]
still_big = [r for r in results if r.get("size_after_mb") and r["size_after_mb"] > CLAUDE_LIMIT_MB]

if failures:
    print(f"\nERRORS ({len(failures)} file(s)):")
    for r in failures:
        print(f"  {r.get('input', r.get('name'))}: {r.get('error')}")

if still_big:
    print(f"\nFiles still over {CLAUDE_LIMIT_MB} MB — lower PYMUPDF_DPI or PYMUPDF_JPEG_QUALITY in the config cell:")
    for r in still_big:
        print(f"  {r.get('input', r.get('name'))}: {r['size_after_mb']:.2f} MB")
else:
    print(f"\nAll processed files are within the {CLAUDE_LIMIT_MB} MB Claude project limit.")

print(f"\nCompressed files saved to: {OUTPUT_DIR}")


[COMPRESS]  20260330_l01_deep-learning.pdf  (22.15 MB) ... 
    ✗ Ghostscript: FAILED — Ghostscript not found
MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor